# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration, processing, and analysis of the FAIR^2 dataset using the `mlcroissant` library, referencing all fields and entities via their `@id` according to best practices.

### Dataset Source
The dataset is structured according to the Croissant schema, accessible via:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

First, we'll load the FAIR^2 dataset's metadata and records via the Croissant schema URL using `mlcroissant`, ensuring all entities are referenced by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Output dataset overview (name, description)
print(metadata.name + ": " + metadata.description)
print("\nDataset identifier (@id):", metadata.id)

## 2. Data Overview

Let's review available record sets and fields. All entities are referenced and listed by their `@id`. If the schema includes multiple record sets, we'll enumerate their `@id` and preview the fields and columns for each.

Note: As per the Croissant schema, record sets are identified by `@id` and provide separate tables (e.g., Table 1: Clinical Features, Table 2: Hormone Measurements, Table 3: Genetic Variants).

In [ ]:
# Get a list of all record set @id's
record_sets = []
for rset in dataset.metadata.record_sets:
    print(f"RecordSet name: {rset.name} | @id: {rset.id}")
    print("Fields:")
    for field in rset.fields:
        print(f"  Field name: {field.name} | @id: {field.id} | Column(s): {[col.id for col in getattr(field, 'columns', [])]}")
    record_sets.append(rset.id)
    print("---")


## 3. Data Extraction

Now, we'll extract the contents of each record set referenced by its `@id` and load them into pandas DataFrames for exploration. All fields/columns are handled via their `@id` as seen above.

In [ ]:
# Extract all record sets as DataFrames using their @id
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFields/columns for record set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate filtering, normalization, and grouping using numeric and categorical fields from the extracted tables, referencing their `@id`s (see the overview above for actual field @id's).

Example: For Table 1 (clinical features), let's filter patients by age at diagnosis (> 25 years), normalize height, and group by presence/absence of infertility (all referenced by `@id`).


In [ ]:
# Assuming Table 1 corresponds to the first record set
record_set_id = record_sets[0]
df = dataframes[record_set_id].copy()

# Find relevant field @id's (example names; replace with actual @id's from the overview)
# Suppose columns include 'age_at_diagnosis', 'height', 'infertility', each with their @id
age_id = [col for col in df.columns if 'age' in col.lower()][0]  # e.g., '@id:age_at_diagnosis'
height_id = [col for col in df.columns if 'height' in col.lower()][0]  # e.g., '@id:height'
infertility_id = [col for col in df.columns if 'infertility' in col.lower()][0]  # e.g., '@id:infertility'

# Filter patients with age > 25
threshold = 25
filtered_df = df[df[age_id] > threshold]
print(f"Filtered records with {age_id} > {threshold}:")
print(filtered_df.head())

# Normalize height
filtered_df[f"{height_id}_normalized"] = (filtered_df[height_id] - filtered_df[height_id].mean()) / filtered_df[height_id].std()
print(f"Normalized {height_id} for filtered records:")
print(filtered_df[[height_id, f"{height_id}_normalized"]].head())

# Group by infertility status
if infertility_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(infertility_id).mean(numeric_only=True)
    print(f"Grouped data by {infertility_id}:")
    print(grouped_df.head())

## 5. Visualization

Let's visualize the distributions and correlations, referencing fields by their `@id`.

Example: plot age vs height and group by infertility status.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Scatter plot: age at diagnosis vs height, colored by infertility status
plt.figure(figsize=(7,4))
sns.scatterplot(
    x=filtered_df[age_id],
    y=filtered_df[height_id],
    hue=filtered_df[infertility_id]
)
plt.title(f"{age_id} vs {height_id}, colored by {infertility_id}")
plt.xlabel(age_id)
plt.ylabel(height_id)
plt.legend(title=infertility_id)
plt.show()

## 6. Conclusion

This notebook demonstrated step-by-step loading and exploration of the FAIR^2 clinical dataset using `mlcroissant`. We:
- Loaded data and accessed entities by their `@id`.
- Reviewed and extracted all tables (record sets).
- Performed filtering, normalization, and grouping based strictly on field/column `@id`.
- Visualized observed clinical features and their relationships.

Due to the limited scope (N=3 patients), further statistical analysis is restricted, yet the process is reproducible for larger datasets with similar Croissant schemas.